# Defining your own dataset

This notebook demonstrates how to optimize over your own labelled dataset. It will include metadata fields on your documents which can be included in your custom defined search queries.

# Dataset

The dataset to reference is `../resources/custom/custom_corpus.json`, with corresponding `../resources/custom/custom_queries.json` and `../resources/custom/custom_qrels.json`. Let's load and inspect them.

In [1]:
import json

corpus_file = "../resources/custom/custom_corpus.json"
queries_file = "../resources/custom/custom_queries.json"
qrels_file = "../resources/custom/custom_qrels.json"

with open(corpus_file) as file:
    corpus = json.load(file)

with open(queries_file) as file:
    queries = json.load(file)

with open(qrels_file) as file:
    qrels = json.load(file)

In [2]:
from redis_retrieval_optimizer.bayes_study import run_bayes_study
from typing import Any
from redisvl.utils.vectorize import BaseVectorizer

config_path = "./custom_data_config.yaml"
redis_url = "redis://localhost:6379"

def custom_corpus_processor(corpus: dict[dict[str:Any]], emb_model: BaseVectorizer) -> list[dict[str:str]]:
    """ Custom processing logic for the corpus
    Args:
        corpus (dict): The corpus data. This is read in from a JSON file containing a single JSON
            object.
        emb_model: The RedisVL embedding model to use for vectorization. These are specified in the
            study_config.yaml file and may be HuggingFace model names, or one of the supported 
            embeddings APIs. see <> for a list.
    Returns:
        list: A list of documents to be indexed in Redis. Each document should have the following
            fields:
            - _id: The unique identifier for the document.

            also have the fields listed under 'additional_fields' defined in the study_config.yaml
            file. In this example they are:
            - product_name: The name of the product.
            - category: The product category, read as a Redis TAG.
            - price: The price of the product.
            - in_stock: A boolean indicating if the product is in stock, cast to string, read as a Redis TAG
            - vector: The vector representation of the product name.
    """
    corpus_data = []

    texts = []
    for key in corpus:
        texts.append(corpus[key]['product_name'])
    embeddings = emb_model.embed_many(texts, as_buffer=True)

    for key, embedding in zip(corpus, embeddings):
        doc = {
            "_id": key,
            "product_name": corpus[key]["product_name"],
            "category": corpus[key]["category"],
            "price": corpus[key]["price"],
            "in_stock": "TRUE" if corpus[key]["in_stock"] else "FALSE",
            "vector_embedding": embedding
        }
        corpus_data.append(doc)

    return corpus_data


/Users/justin.cechmanek/.pyenv/versions/3.11.9/envs/ret-opt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# because we are defining a custom search method that will utilize RedisVL FilterExpressions
# we need to define a custom search method map that tells the retrieval optimizer how to
# handle the custom search method.

from redis_retrieval_optimizer.schema import SearchMethodInput, SearchMethodOutput

from redis_retrieval_optimizer.search_methods import (
    gather_vector_results,
    gather_hybrid_results,
    gather_bm25_results,
)


def custom_gather_filter_query_results( search_method_input: SearchMethodInput) -> SearchMethodOutput:
    """ Custom search method that uses RedisVL FilterExpressions to filter results based on the
    query.
    Args:
        search_method_input (SearchMethodInput): The input to the search method, which includes
        the query, corpus, and Redis URL.
    Returns:
        SearchMethodOutput
    """

    return gather_vector_results(search_method_input)
    # This function will use RedisVL FilterExpressions to filter results based on the query.



custom_search_method_map = {
    "vector": gather_vector_results, # this is predefined, so we can just import and ust it
    "hybrid": gather_hybrid_results,
    "bm25": gather_bm25_results,
    "filter_query": custom_gather_filter_query_results # this is our custom search method
    }


In [4]:
# run the study
metrics = run_bayes_study(
    config_path=config_path,
    redis_url=redis_url,
    #corpus_processor=eval_beir.process_corpus,
    corpus_processor=custom_corpus_processor,
    search_method_map=custom_search_method_map,
)

[I 2026-07-17 13:07:12,217] A new study created in memory with name: test
[I 2026-07-17 13:07:39,060] Trial 0 finished with value: -0.4999951214723237 and parameters: {'model_info': {'type': 'hf', 'model': 'sentence-transformers/all-MiniLM-L6-v2', 'dim': 384, 'embedding_cache_name': 'vec-cache', 'dtype': 'float32'}, 'search_method': 'bm25', 'algorithm': 'hnsw', 'var_dtype': 'float32', 'distance_metric': 'cosine', 'ret_k': 5, 'ef_runtime': 10, 'ef_construction': 300, 'm': 8}. Best is trial 0 with value: -0.4999951214723237.
[I 2026-07-17 13:07:40,536] Trial 1 finished with value: -0.4999951214723237 and parameters: {'model_info': {'type': 'hf', 'model': 'sentence-transformers/all-MiniLM-L6-v2', 'dim': 384, 'embedding_cache_name': 'vec-cache', 'dtype': 'float32'}, 'search_method': 'bm25', 'algorithm': 'hnsw', 'var_dtype': 'float32', 'distance_metric': 'cosine', 'ret_k': 10, 'ef_runtime': 10, 'ef_construction': 300, 'm': 8}. Best is trial 0 with value: -0.4999951214723237.
[I 2026-07-17 1

In [5]:
metrics.sort_values(by='search_method')

,search_method,total_indexing_time,avg_query_time,model,model_dim,ret_k,recall,ndcg,f1,precision,algorithm,ef_construction,ef_runtime,m,distance_metric,vector_data_type,objective_value,total_memory_mb
0,bm25,0.003989,0.000649,sentence-transformers/all-MiniLM-L6-v2,384,5,0.472143,0.50751,0.496032,0.595833,hnsw,300,10,8,cosine,float32,-0.499995,0.08646
1,bm25,0.003989,0.000448,sentence-transformers/all-MiniLM-L6-v2,384,10,0.472143,0.50751,0.496032,0.595833,hnsw,300,10,8,cosine,float32,-0.499995,0.08646
2,bm25,0.003989,0.000447,sentence-transformers/all-MiniLM-L6-v2,384,9,0.472143,0.50751,0.496032,0.595833,hnsw,100,50,8,cosine,float32,-0.499995,0.08646
3,bm25,0.003989,0.000668,sentence-transformers/all-MiniLM-L6-v2,384,6,0.472143,0.50751,0.496032,0.595833,hnsw,300,10,8,cosine,float32,-0.499995,0.08646
4,bm25,0.003989,0.000425,sentence-transformers/all-MiniLM-L6-v2,384,10,0.472143,0.50751,0.496032,0.595833,hnsw,300,30,8,cosine,float32,-0.499995,0.08646
5,bm25,0.003989,0.000559,sentence-transformers/all-MiniLM-L6-v2,384,9,0.472143,0.50751,0.496032,0.595833,hnsw,300,10,16,cosine,float32,-0.499995,0.08646
6,bm25,0.003989,0.000387,sentence-transformers/all-MiniLM-L6-v2,384,7,0.472143,0.50751,0.496032,0.595833,hnsw,100,50,64,cosine,float32,-0.499995,0.08646
7,bm25,0.003989,0.000448,sentence-transformers/all-MiniLM-L6-v2,384,5,0.472143,0.50751,0.496032,0.595833,hnsw,100,10,8,cosine,float32,-0.499995,0.08646
8,bm25,0.003989,0.000563,sentence-transformers/all-MiniLM-L6-v2,384,5,0.472143,0.50751,0.496032,0.595833,hnsw,300,10,8,cosine,float32,-0.499995,0.08646
9,bm25,0.003989,0.000561,sentence-transformers/all-MiniLM-L6-v2,384,6,0.472143,0.50751,0.496032,0.595833,hnsw,300,30,64,cosine,float32,-0.499995,0.08646
